# monogate Mathematical Explorer

**Self-improving conjecture discovery**: the EMLProverV2 generates candidate
mathematical identities, proves them via the 4-tier pipeline, compresses
the proofs, tests adversarial variants, and ranks discoveries by
**interestingness = confidence × elegance × novelty**.

Each round the neural scorer learns from successful proofs, making future
MCTS searches faster. The discovered catalog feeds back into generation,
steering mutations toward genuinely new territory.

---
**What you get from this notebook:**
- Interactive parameter panel (category, rounds, budget, temperature)
- Live exploration log
- Ranked table of discovered identities with elegance / novelty / proof sketch
- 3-panel session visualization (timeline, breakdown, scorer learning)
- Gallery of the top-5 most interesting discoveries

In [ ]:
import math, warnings, sys
warnings.filterwarnings('ignore')

from monogate.prover import (
    EMLProverV2,
    interestingness_score,
    elegance_score,
    novelty_score,
)
from monogate.identities import ALL_IDENTITIES
import matplotlib.pyplot as plt

prover = EMLProverV2(enable_learning=True, verbose=False)
print(f'monogate Mathematical Explorer')
print(f'  Seed catalog: {len(ALL_IDENTITIES)} identities')
print(f'  Neural scorer: {"enabled" if prover.scorer else "disabled"}')
print(f'  Prover version: EMLProverV2')

## Parameters

Adjust the sliders below, then run **Cell 3** to start the explorer.

| Parameter | Meaning |
|-----------|--------|
| **Category** | Seed identity category for mutations |
| **Rounds** | Number of generate→verify→learn cycles |
| **Budget/round** | Conjectures generated per round |
| **Temperature** | Mutation aggressiveness (0.3=safe, 1.0=creative) |

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    cat_w   = widgets.Dropdown(
        options=['trig', 'hyperbolic', 'exponential', 'special', 'physics', 'eml'],
        value='trig', description='Category:', style={'description_width': 'initial'}
    )
    rounds_w = widgets.IntSlider(
        min=2, max=20, value=8, description='Rounds:',
        style={'description_width': 'initial'}
    )
    budget_w = widgets.IntSlider(
        min=20, max=300, value=80, step=20, description='Budget/round:',
        style={'description_width': 'initial'}
    )
    temp_w  = widgets.FloatSlider(
        min=0.3, max=1.0, step=0.05, value=0.75, description='Temperature:',
        style={'description_width': 'initial'}, readout_format='.2f'
    )
    display(cat_w, rounds_w, budget_w, temp_w)
    CATEGORY    = cat_w
    N_ROUNDS    = rounds_w
    BUDGET      = budget_w
    TEMPERATURE = temp_w
    print('Use the widgets above, then run Cell 3.')
except ImportError:
    class _V:
        def __init__(self, v): self.value = v
    CATEGORY    = _V('trig')
    N_ROUNDS    = _V(8)
    BUDGET      = _V(80)
    TEMPERATURE = _V(0.75)
    print('ipywidgets not available — using defaults:')
    print(f'  category={CATEGORY.value}, rounds={N_ROUNDS.value}, '
          f'budget={BUDGET.value}, temperature={TEMPERATURE.value}')

## Exploration Run

Each round: generate conjectures → 4-tier proof → compress witness → adversarial test → neural scorer update

In [ ]:
print(f'Starting exploration: category={CATEGORY.value}, '
      f'rounds={N_ROUNDS.value}, budget={BUDGET.value}, '
      f'temperature={TEMPERATURE.value}')
print()

# Re-enable verbose for live feedback
prover.verbose = True

session = prover.explore(
    n_rounds         = N_ROUNDS.value,
    n_per_round      = BUDGET.value,
    seed_category    = CATEGORY.value,
    temperature      = TEMPERATURE.value,
    compress_witnesses = True,
)

prover.verbose = False
n_disc = session['n_total_discovered']
print(f'\nExploration complete: {n_disc} identities discovered')

## Ranking Table

Discovered identities ranked by **interestingness = confidence × elegance × novelty**.

In [ ]:
try:
    import pandas as pd
    _pd_ok = True
except ImportError:
    _pd_ok = False

rows = []
for conj, result in session['discovered']:
    nc  = result.node_count or max(1, len(conj.expression) // 10)
    elg = elegance_score(nc)
    nov = novelty_score(conj.expression, ALL_IDENTITIES)
    ist = interestingness_score(result.confidence, nc, conj.expression, ALL_IDENTITIES)
    rows.append({
        'Identity':       conj.expression[:70],
        'Method':         result.verification_method,
        'Conf':           f'{result.confidence:.2f}',
        'Nodes':          nc,
        'Elegance':       f'{elg:.4f}',
        'Novelty':        f'{nov:.4f}',
        'Interestingness':f'{ist:.4f}',
        'Proof':          EMLProverV2._proof_sketch(result)[:60],
    })

if not rows:
    print('No identities discovered this session.')
elif _pd_ok:
    df = pd.DataFrame(rows).sort_values('Interestingness', ascending=False)
    pd.set_option('display.max_colwidth', 80)
    pd.set_option('display.width', 200)
    print(df.to_string(index=False))
else:
    rows_sorted = sorted(rows, key=lambda r: r['Interestingness'], reverse=True)
    for r in rows_sorted:
        print(f"  [{r['Interestingness']}] {r['Identity'][:60]}")
        print(f"     → {r['Proof']}")

## Session Visualization

Three panels: discovery timeline, per-round breakdown, and scorer learning curve.

In [ ]:
%matplotlib inline
prover.visualize_explorer_session(session)

## Gallery: Top-5 Discoveries

The most interesting identities found this session, ranked by **interestingness**.

In [ ]:
try:
    from IPython.display import Latex, display as ipy_display
    _ipy_ok = True
except ImportError:
    _ipy_ok = False

if not session['discovered']:
    print('No discoveries to display.')
else:
    top5 = sorted(
        session['discovered'],
        key=lambda cr: interestingness_score(
            cr[1].confidence,
            cr[1].node_count or 5,
            cr[0].expression,
            ALL_IDENTITIES,
        ),
        reverse=True,
    )[:5]

    for i, (conj, result) in enumerate(top5, 1):
        nc  = result.node_count or 5
        ist = interestingness_score(
            result.confidence, nc, conj.expression, ALL_IDENTITIES
        )
        print(f'\n{"="*64}')
        print(f'  #{i}  {conj.name}')
        print(f'  Expression:      {conj.expression}')
        print(f'  Proof:           {EMLProverV2._proof_sketch(result)}')
        print(f'  Interestingness: {ist:.4f}')
        print(f'  Elegance:        {elegance_score(nc):.4f}  ({nc} nodes)')
        print(f'  Novelty:         {novelty_score(conj.expression, ALL_IDENTITIES):.4f}')
        if _ipy_ok and conj.latex:
            try:
                ipy_display(Latex(f'$${conj.latex}$$'))
            except Exception:
                print(f'  LaTeX: {conj.latex}')
    print(f'\n{"="*64}')

## Bonus: Grammar Extension Frontier

The **Physics-Completeness Theorem** experiment: does adding `−x` as a leaf
make EML physics-complete?

Run with `--quick` for a fast smoke test, or remove `--quick` for full results.

In [ ]:
from monogate.frontiers.grammar_extension import run_grammar_extension

grammar_results = run_grammar_extension(n_simulations=500, verbose=True)
barrier = grammar_results['barrier']

print(f'\nSummary:')
print(f'  Standard native: {barrier["n_native_standard"]}/{barrier["n_laws"]}')
print(f'  Extended native: {barrier["n_native_extended"]}/{barrier["n_laws"]}')
print(f'  Newly reachable: {barrier["newly_reachable"]}')